# SQL practice 10: changing data: DML, MERGE and history

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-10-changing-data.ipynb)

13 exercises on INSERT, UPDATE, DELETE, MERGE patterns, partition replacement, SCD2 and time travel. Each exercise names the [reference entries](sql-reference-10-changing-data.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 10](sql-reference-10-changing-data.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. This notebook uses a Delta Lake session (the Delta jars download on first use), because `UPDATE`, `DELETE` and `MERGE` need a table format that supports them. `fresh(table, select)` drops and re-creates a Delta table.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="sql-")  # tables created in this notebook live here

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, UPDATE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

def fresh(table, select):
    """Drop and re-create a Delta table from a query, so its history starts again at version 0."""
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    spark.sql(f"CREATE TABLE {table} USING DELTA AS {select}")

## Exercises

4 easy, 8 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Append a day of orders](#exercise-1-append-a-day-of-orders) | Easy | [`CREATE TABLE AS SELECT` and `INSERT`](sql-reference-10-changing-data.ipynb#create-table-as-select-and-insert) |
| 2 | [Correct a status in place](#exercise-2-correct-a-status-in-place) | Easy | [`UPDATE` and `DELETE`](sql-reference-10-changing-data.ipynb#update-and-delete) |
| 3 | [Delete rows matching another table](#exercise-3-delete-rows-matching-another-table) | Medium | [`UPDATE` and `DELETE`](sql-reference-10-changing-data.ipynb#update-and-delete), [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert) |
| 4 | [Upsert customer changes](#exercise-4-upsert-customer-changes) | Medium | [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert) |
| 5 | [Ignore late, older updates](#exercise-5-ignore-late-older-updates) | Medium | [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert) |
| 6 | [De-duplicate the source before merging](#exercise-6-de-duplicate-the-source-before-merging) | Medium | [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert) |
| 7 | [Apply deletes from a change feed](#exercise-7-apply-deletes-from-a-change-feed) | Medium | [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert) |
| 8 | [Synchronise with a full snapshot](#exercise-8-synchronise-with-a-full-snapshot) | Medium | [Synchronising a table with a full snapshot](sql-reference-10-changing-data.ipynb#synchronising-a-table-with-a-full-snapshot) |
| 9 | [Replace one day idempotently](#exercise-9-replace-one-day-idempotently) | Medium | [`CREATE TABLE AS SELECT` and `INSERT`](sql-reference-10-changing-data.ipynb#create-table-as-select-and-insert) |
| 10 | [A summary table with CTAS](#exercise-10-a-summary-table-with-ctas) | Easy | [`CREATE TABLE AS SELECT` and `INSERT`](sql-reference-10-changing-data.ipynb#create-table-as-select-and-insert) |
| 11 | [Read an earlier version](#exercise-11-read-an-earlier-version) | Easy | [Transactions, time travel and restore](sql-reference-10-changing-data.ipynb#transactions-time-travel-and-restore) |
| 12 | [Running the same MERGE twice](#exercise-12-running-the-same-merge-twice) | Medium | [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert) |
| 13 | [SCD type 2 in one MERGE](#exercise-13-scd-type-2-in-one-merge) | Hard | [Slowly changing dimensions type 2 with `MERGE`](sql-reference-10-changing-data.ipynb#slowly-changing-dimensions-type-2-with-merge) |

## Exercise 1: Append a day of orders

**Difficulty:** Easy · about 5 min · **Practises:** [`CREATE TABLE AS SELECT` and `INSERT`](sql-reference-10-changing-data.ipynb#create-table-as-select-and-insert)

*Daily loads append a slice of the source.*

`fact_orders` holds orders up to 2026-01-09. Append the orders from `orders` placed on 2026-01-10 or later, then return `order_date` and `orders` per day from 2026-01-09 on, ordered by `order_date`.

In [ ]:
def reset():
    fresh("fact_orders", "SELECT order_id, customer_id, status, to_date(order_ts) AS order_date, amount FROM orders WHERE order_ts < TIMESTAMP'2026-01-10 00:00:00'")
reset()
show("SELECT max(order_date) AS loaded_until, count(*) AS rows FROM fact_orders")

**Expected output**

| order_date | orders |
| --- | --- |
| 2026-01-09 | 1 |
| 2026-01-10 | 1 |
| 2026-01-12 | 2 |
| 2026-01-13 | 1 |
| 2026-01-14 | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    INSERT INTO fact_orders
    SELECT order_id, customer_id, status, to_date(order_ts) AS order_date, amount
    FROM orders
    WHERE order_ts >= TIMESTAMP'2026-01-10 00:00:00'
""")
q = """
    SELECT order_date, count(*) AS orders
    FROM fact_orders
    WHERE order_date >= DATE'2026-01-09'
    GROUP BY order_date
    ORDER BY order_date
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Correct a status in place

**Difficulty:** Easy · about 5 min · **Practises:** [`UPDATE` and `DELETE`](sql-reference-10-changing-data.ipynb#update-and-delete)

*Back-office corrections update rows rather than reloading the table.*

In `fact_orders`, mark every `placed` order older than 2026-01-10 as `cancelled`, then return `status` and `orders`, ordered by `status`.

In [ ]:
def reset():
    fresh("fact_orders", "SELECT order_id, customer_id, status, to_date(order_ts) AS order_date, amount FROM orders")
reset()

**Expected output**

| status | orders |
| --- | --- |
| cancelled | 1 |
| paid | 4 |
| placed | 1 |
| returned | 2 |
| shipped | 4 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    UPDATE fact_orders SET status = 'cancelled' WHERE status = 'placed' AND order_date < DATE'2026-01-10'
""")
q = """
    SELECT status, count(*) AS orders
    FROM fact_orders
    GROUP BY status
    ORDER BY status
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Delete rows matching another table

**Difficulty:** Medium · about 5 min · **Practises:** [`UPDATE` and `DELETE`](sql-reference-10-changing-data.ipynb#update-and-delete), [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert)

*Erasure requests remove every row for a set of customers; Delta needs MERGE for a subquery-driven delete.*

Delete from `fact_orders` all orders of customers in the West region (look the region up in `customers`), then return `customer_id` and `orders` for the remaining rows, ordered by `customer_id`.

In [ ]:
def reset():
    fresh("fact_orders", "SELECT order_id, customer_id, status, to_date(order_ts) AS order_date, amount FROM orders")
reset()

**Expected output**

| customer_id | orders |
| --- | --- |
| 1 | 3 |
| 2 | 2 |
| 3 | 2 |
| 4 | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO fact_orders t
    USING (SELECT customer_id FROM customers WHERE region = 'West') s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN DELETE
""")
q = """
    SELECT customer_id, count(*) AS orders
    FROM fact_orders
    GROUP BY customer_id
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Upsert customer changes

**Difficulty:** Medium · about 10 min · **Practises:** [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert)

*The bread-and-butter MERGE of every dimension load.*

Merge `customer_changes` (below) into `dim_customer` on `customer_id`: update `name`, `region` and `updated_at` of existing customers, insert new ones. Return all rows of `dim_customer` ordered by `customer_id`.

In [ ]:
def reset():
    fresh("dim_customer", "SELECT customer_id, name, region, DATE'2026-01-01' AS updated_at FROM customers WHERE customer_id <= 5")
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW customer_changes AS SELECT * FROM VALUES
      (2, 'Bright Retail Ltd', 'North', DATE'2026-01-10'), (5, 'Echo Supplies', 'South', DATE'2026-01-11'),
      (9, 'Iris Imports', 'East', DATE'2026-01-12')
    AS c(customer_id, name, region, updated_at)
    """)
reset()

**Expected output**

| customer_id | name | region | updated_at |
| --- | --- | --- | --- |
| 1 | Acme Corp | North | 2026-01-01 |
| 2 | Bright Retail Ltd | North | 2026-01-10 |
| 3 | Cyan Traders | South | 2026-01-01 |
| 4 | Delta Grocers | null | 2026-01-01 |
| 5 | Echo Supplies | South | 2026-01-11 |
| 9 | Iris Imports | East | 2026-01-12 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO dim_customer t
    USING customer_changes s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET name = s.name, region = s.region, updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT (customer_id, name, region, updated_at) VALUES (s.customer_id, s.name, s.region, s.updated_at)
""")
q = """
    SELECT customer_id, name, region, updated_at
    FROM dim_customer
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Ignore late, older updates

**Difficulty:** Medium · about 10 min · **Practises:** [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert)

*Out-of-order delivery must not overwrite newer data with older data.*

Merge `late_changes` (below) into `dim_customer`, updating a customer only when the incoming `updated_at` is newer than the stored one, and inserting new customers. Return all rows ordered by `customer_id`.

In [ ]:
def reset():
    fresh("dim_customer", "SELECT customer_id, name, region, DATE'2026-01-01' AS updated_at FROM customers WHERE customer_id <= 5")
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW late_changes AS SELECT * FROM VALUES
      (1, 'Acme Corp', 'South', DATE'2025-12-01'), (3, 'Cyan Traders', 'East', DATE'2026-01-05'),
      (8, 'Hotel Homeware', 'East', DATE'2025-11-05')
    AS c(customer_id, name, region, updated_at)
    """)
reset()

**Expected output**

| customer_id | region | updated_at |
| --- | --- | --- |
| 1 | North | 2026-01-01 |
| 2 | North | 2026-01-01 |
| 3 | East | 2026-01-05 |
| 4 | null | 2026-01-01 |
| 5 | West | 2026-01-01 |
| 8 | East | 2025-11-05 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO dim_customer t
    USING late_changes s
    ON t.customer_id = s.customer_id
    WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET name = s.name, region = s.region, updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT (customer_id, name, region, updated_at) VALUES (s.customer_id, s.name, s.region, s.updated_at)
""")
q = """
    SELECT customer_id, region, updated_at
    FROM dim_customer
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: De-duplicate the source before merging

**Difficulty:** Medium · about 15 min · **Practises:** [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert)

*A batch with several versions of one key makes MERGE fail.*

`cdc_batch` (below) contains several versions of some customers. Merge only the latest version of each `customer_id` (by `updated_at`) into `dim_customer` (update or insert). Return all rows ordered by `customer_id`.

In [ ]:
def reset():
    fresh("dim_customer", "SELECT customer_id, name, region, DATE'2026-01-01' AS updated_at FROM customers WHERE customer_id <= 5")
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW cdc_batch AS SELECT * FROM VALUES
      (2, 'Bright Retail', 'South', DATE'2026-01-10'), (2, 'Bright Retail', 'East', DATE'2026-01-12'),
      (6, 'Foxtrot Foods', 'North', DATE'2026-01-11'), (6, 'Foxtrot Foods', 'West', DATE'2026-01-09')
    AS c(customer_id, name, region, updated_at)
    """)
reset()

**Expected output**

| customer_id | region | updated_at |
| --- | --- | --- |
| 1 | North | 2026-01-01 |
| 2 | East | 2026-01-12 |
| 3 | South | 2026-01-01 |
| 4 | null | 2026-01-01 |
| 5 | West | 2026-01-01 |
| 6 | North | 2026-01-11 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO dim_customer t
    USING (
        SELECT customer_id, name, region, updated_at
        FROM (SELECT *, row_number() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn FROM cdc_batch)
        WHERE rn = 1
    ) s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET name = s.name, region = s.region, updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT (customer_id, name, region, updated_at) VALUES (s.customer_id, s.name, s.region, s.updated_at)
""")
q = """
    SELECT customer_id, region, updated_at
    FROM dim_customer
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Apply deletes from a change feed

**Difficulty:** Medium · about 10 min · **Practises:** [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert)

*CDC feeds carry an operation flag; one MERGE applies inserts, updates and deletes.*

Merge `changes` (below, `op` is `I`, `U` or `D`) into `dim_customer`: delete on `D`, update on `U` (and on `I` if the key exists), insert on `I`/`U` when the key is new. Return all rows ordered by `customer_id`.

In [ ]:
def reset():
    fresh("dim_customer", "SELECT customer_id, name, region, DATE'2026-01-01' AS updated_at FROM customers WHERE customer_id <= 5")
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW changes AS SELECT * FROM VALUES
      ('D', 1, NULL, NULL), ('U', 2, 'Bright Retail', 'West'), ('I', 7, 'Gamma Sports', 'East'), ('D', 99, NULL, NULL)
    AS c(op, customer_id, name, region)
    """)
reset()

**Expected output**

| customer_id | name | region |
| --- | --- | --- |
| 2 | Bright Retail | West |
| 3 | Cyan Traders | South |
| 4 | Delta Grocers | null |
| 5 | Echo Supplies | West |
| 7 | Gamma Sports | East |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO dim_customer t
    USING changes s
    ON t.customer_id = s.customer_id
    WHEN MATCHED AND s.op = 'D' THEN DELETE
    WHEN MATCHED THEN UPDATE SET name = s.name, region = s.region
    WHEN NOT MATCHED AND s.op <> 'D' THEN INSERT (customer_id, name, region, updated_at) VALUES (s.customer_id, s.name, s.region, DATE'2026-01-15')
""")
q = """
    SELECT customer_id, name, region
    FROM dim_customer
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Synchronise with a full snapshot

**Difficulty:** Medium · about 10 min · **Practises:** [Synchronising a table with a full snapshot](sql-reference-10-changing-data.ipynb#synchronising-a-table-with-a-full-snapshot)

*Small reference tables are often re-sent in full; the target must match exactly.*

Make `dim_product` identical to `product_snapshot` (below) with one MERGE: update changed rows, insert new ones, delete rows missing from the snapshot. Return all rows ordered by `sku`.

In [ ]:
def reset():
    fresh("dim_product", "SELECT sku, category, list_price FROM products WHERE sku <= 'SKU-4005'")
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW product_snapshot AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
      ('SKU-4001', 'Kitchen', 13.00), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4004', 'Outdoor', 55.50), ('SKU-4020', 'Pantry', 4.20)
    AS s(sku, category, list_price)
    """)
reset()

**Expected output**

| sku | category | list_price |
| --- | --- | --- |
| SKU-4001 | Kitchen | 13.00 |
| SKU-4002 | Kitchen | 34.50 |
| SKU-4004 | Outdoor | 55.50 |
| SKU-4020 | Pantry | 4.20 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO dim_product t
    USING product_snapshot s
    ON t.sku = s.sku
    WHEN MATCHED AND (t.category <> s.category OR t.list_price <> s.list_price) THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
q = """
    SELECT sku, category, list_price
    FROM dim_product
    ORDER BY sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Replace one day idempotently

**Difficulty:** Medium · about 10 min · **Practises:** [`CREATE TABLE AS SELECT` and `INSERT`](sql-reference-10-changing-data.ipynb#create-table-as-select-and-insert)

*Re-processing a day must replace it, not duplicate it.*

`daily_revenue` already contains a wrong row for 2026-01-12. Replace that day's rows with the correct totals computed from `orders` (one row per status) using `INSERT ... REPLACE WHERE`. Return all rows for 2026-01-11 to 2026-01-13, ordered by `day`, `status`.

In [ ]:
def reset():
    fresh("daily_revenue", """
        SELECT to_date(order_ts) AS day, status, sum(amount) AS revenue FROM orders
        WHERE to_date(order_ts) <> DATE'2026-01-12' GROUP BY 1, 2
        UNION ALL SELECT DATE'2026-01-12', 'paid', CAST(999.00 AS DECIMAL(22,2))
    """)
reset()

**Expected output**

| day | status | revenue |
| --- | --- | --- |
| 2026-01-12 | paid | 74.00 |
| 2026-01-12 | shipped | 1180.00 |
| 2026-01-13 | placed | 210.00 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    INSERT INTO daily_revenue REPLACE WHERE day = DATE'2026-01-12'
    SELECT to_date(order_ts) AS day, status, sum(amount) AS revenue
    FROM orders
    WHERE to_date(order_ts) = DATE'2026-01-12'
    GROUP BY to_date(order_ts), status
""")
q = """
    SELECT day, status, revenue
    FROM daily_revenue
    WHERE day BETWEEN DATE'2026-01-11' AND DATE'2026-01-13'
    ORDER BY day, status
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: A summary table with CTAS

**Difficulty:** Easy · about 5 min · **Practises:** [`CREATE TABLE AS SELECT` and `INSERT`](sql-reference-10-changing-data.ipynb#create-table-as-select-and-insert)

*Materialising an aggregate for dashboards.*

Create (or replace) the Delta table `customer_summary` with `customer_id`, `orders`, `revenue` and `last_order` per customer from `orders`, then return its rows where `orders >= 2`, ordered by `revenue` descending.

**Expected output**

| customer_id | orders | revenue | last_order |
| --- | --- | --- | --- |
| 4 | 2 | 1269.90 | 2026-01-12 |
| 3 | 2 | 459.00 | 2026-01-09 |
| 2 | 2 | 315.25 | 2026-01-07 |
| 6 | 2 | 306.40 | 2026-01-14 |
| 1 | 3 | 234.50 | 2026-01-05 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
spark.sql("""
    CREATE OR REPLACE TABLE customer_summary USING DELTA AS
    SELECT customer_id, count(*) AS orders, sum(amount) AS revenue, to_date(max(order_ts)) AS last_order
    FROM orders
    GROUP BY customer_id
""")
q = """
    SELECT customer_id, orders, revenue, last_order
    FROM customer_summary
    WHERE orders >= 2
    ORDER BY revenue DESC
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Read an earlier version

**Difficulty:** Easy · about 5 min · **Practises:** [Transactions, time travel and restore](sql-reference-10-changing-data.ipynb#transactions-time-travel-and-restore)

*Time travel answers "what did this table look like before the load?".*

`audit_orders` was created (version 0) and then updated and deleted from (below). Return `version` (`v0` or `now`), `order_id` and `status` from version 0 and from the current version, ordered by `version` descending (`v0` first), then `order_id`.

In [ ]:
fresh("audit_orders", "SELECT order_id, status FROM orders WHERE order_id <= 4")
spark.sql("UPDATE audit_orders SET status = 'cancelled' WHERE order_id = 2")
spark.sql("DELETE FROM audit_orders WHERE order_id = 4")
spark.sql("DESCRIBE HISTORY audit_orders").select("version", "operation").orderBy("version").show(truncate=False)   # a command, not a subquery

**Expected output**

| version | order_id | status |
| --- | --- | --- |
| v0 | 1 | paid |
| v0 | 2 | shipped |
| v0 | 3 | returned |
| v0 | 4 | paid |
| now | 1 | paid |
| now | 2 | cancelled |
| now | 3 | returned |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT 'v0' AS version, order_id, status FROM audit_orders VERSION AS OF 0
    UNION ALL
    SELECT 'now' AS version, order_id, status FROM audit_orders
    ORDER BY version DESC, order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Running the same MERGE twice

**Difficulty:** Medium · about 10 min · **Practises:** [`MERGE INTO` (upsert)](sql-reference-10-changing-data.ipynb#merge-into-upsert)

*Retries are safe only when the load is idempotent.*

Run the de-duplicated upsert of `retry_batch` (below) into `dim_customer` **twice** (as a retried job would), then return `rows` (total rows) and `max_updated` from `dim_customer`. The result must equal running it once.

In [ ]:
def reset():
    fresh("dim_customer", "SELECT customer_id, name, region, DATE'2026-01-01' AS updated_at FROM customers WHERE customer_id <= 5")
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW retry_batch AS SELECT * FROM VALUES
      (5, 'Echo Supplies', 'North', DATE'2026-01-20'), (10, 'Juno Tools', 'West', DATE'2026-01-20')
    AS c(customer_id, name, region, updated_at)
    """)
reset()

**Expected output**

| rows | max_updated |
| --- | --- |
| 6 | 2026-01-20 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO dim_customer t USING retry_batch s ON t.customer_id = s.customer_id
    WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("""
    MERGE INTO dim_customer t USING retry_batch s ON t.customer_id = s.customer_id
    WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
q = """
    SELECT count(*) AS rows, max(updated_at) AS max_updated
    FROM dim_customer
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: SCD type 2 in one MERGE

**Difficulty:** Hard · about 20 min · **Practises:** [Slowly changing dimensions type 2 with `MERGE`](sql-reference-10-changing-data.ipynb#slowly-changing-dimensions-type-2-with-merge)

*History-preserving dimensions are a staple of data-engineering interviews.*

`dim_region_history` tracks each customer's region with `valid_from`, `valid_to` and `is_current`. Apply `region_changes` (below) as of 2026-01-15: close the current row of customers whose region changed and insert their new current row; insert new customers; leave unchanged customers alone. Return all rows ordered by `customer_id`, `valid_from`.

In [ ]:
def reset():
    fresh("dim_region_history", """
        SELECT customer_id, region, DATE'2025-01-01' AS valid_from, CAST(NULL AS DATE) AS valid_to, true AS is_current
        FROM customers WHERE customer_id <= 3
    """)
    spark.sql("""
    CREATE OR REPLACE TEMP VIEW region_changes AS SELECT * FROM VALUES
      (1, 'North'), (2, 'South'), (9, 'East')
    AS c(customer_id, region)
    """)
reset()

**Expected output**

| customer_id | region | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- |
| 1 | North | 2025-01-01 | null | true |
| 2 | North | 2025-01-01 | 2026-01-15 | false |
| 2 | South | 2026-01-15 | null | true |
| 3 | South | 2025-01-01 | null | true |
| 9 | East | 2026-01-15 | null | true |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
reset()  # start from the same data every time you run this cell
spark.sql("""
    MERGE INTO dim_region_history t
    USING (
        SELECT c.customer_id AS merge_key, c.customer_id, c.region FROM region_changes c
        UNION ALL
        SELECT NULL AS merge_key, c.customer_id, c.region
        FROM region_changes c
        JOIN dim_region_history d ON d.customer_id = c.customer_id AND d.is_current AND d.region <> c.region
    ) s
    ON t.customer_id = s.merge_key AND t.is_current
    WHEN MATCHED AND t.region <> s.region THEN UPDATE SET valid_to = DATE'2026-01-15', is_current = false
    WHEN NOT MATCHED THEN INSERT (customer_id, region, valid_from, valid_to, is_current)
                          VALUES (s.customer_id, s.region, DATE'2026-01-15', NULL, true)
""")
q = """
    SELECT customer_id, region, valid_from, valid_to, is_current
    FROM dim_region_history
    ORDER BY customer_id, valid_from
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 10: changing data: DML, MERGE and history](sql-reference-10-changing-data.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).